In [3]:
!pip install wikipedia-api

^C



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import wikipediaapi
import json
import os

def process_historical_figures(input_file, output_file):
    # Khởi tạo Wikipedia API với ngôn ngữ tiếng Việt
    # Lưu ý: Wikipedia yêu cầu khai báo User-Agent rõ ràng
    wiki = wikipediaapi.Wikipedia(
        user_agent='HistoricalDataBot/1.0 (contact@yourdomain.com)',
        language='vi',
        extract_format=wikipediaapi.ExtractFormat.WIKI
    )

    results = []

    # Kiểm tra nếu file input tồn tại
    if not os.path.exists(input_file):
        print(f"Lỗi: Không tìm thấy file {input_file}")
        return

    with open(input_file, 'r', encoding='utf-8') as f:
        names = [line.strip() for line in f if line.strip()]

    for name in names:
        print(f"Đang xử lý: {name}...")
        page = wiki.page(name)

        if page.exists():
            # Cấu trúc hóa dữ liệu
            data = {
                "name": name,
                "title": page.title,
                "summary": page.summary,
                "full_content": page.text,
                "url": page.fullurl,
                "sections": []
            }

            # Lấy chi tiết từng mục lớn (ví dụ: Thân thế, Sự nghiệp)
            for section in page.sections:
                data["sections"].append({
                    "title": section.title,
                    "text": section.text
                })

            results.append(data)
        else:
            print(f"(!) Không tìm thấy trang Wikipedia cho: {name}")

    # Ghi dữ liệu ra file JSON
    with open(output_file, 'w', encoding='utf-8') as jf:
        json.dump(results, jf, ensure_ascii=False, indent=4)

    print(f"\nHoàn thành! Đã lưu dữ liệu vào {output_file}")

# Thực thi
if __name__ == "__main__":
    # Tên file đầu vào (txt) và đầu ra (json)
    input_path = "danh_sach_nhan_vat.txt"
    output_path = "individual.json"
    
    process_historical_figures(input_path, output_path)

Đang xử lý: Đinh Tiên Hoàng...
Đang xử lý: Hồ Chí Minh...
Đang xử lý: Hai Bà Trưng...
Đang xử lý: Hùng Vương...
Đang xử lý: Lê Thái Tổ...
Đang xử lý: Lê Đại Hành...
Đang xử lý: Lý Nam Đế...
Đang xử lý: Lý Thái Tổ...
Đang xử lý: Lý Thường Kiệt...
Đang xử lý: Quang Trung...
Đang xử lý: Ngô Quyền...
Đang xử lý: Nguyễn Trãi...
Đang xử lý: Trần Hưng Đạo...
Đang xử lý: Trần Nhân Tông...

Hoàn thành! Đã lưu dữ liệu vào individual.json


In [4]:
import wikipediaapi
import json
import os
import re

def clean_extra_whitespace(text):
    """Làm sạch khoảng trắng thừa"""
    return re.sub(r'\s+', ' ', text).strip()

def get_all_sections(sections):
    """Hàm đệ quy để lấy toàn bộ nội dung các mục con"""
    sect_data = []
    for s in sections:
        sect_data.append({
            "title": s.title,
            "text": s.text.strip(),
            "subsections": get_all_sections(s.sections) # Đệ quy lấy mục con (nếu có)
        })
    return sect_data

def process_historical_figures(input_file, output_file):
    wiki = wikipediaapi.Wikipedia(
        user_agent='HistoricalDataBot/1.0 (contact@yourdomain.com)',
        language='vi',
        extract_format=wikipediaapi.ExtractFormat.WIKI
    )

    results = []

    if not os.path.exists(input_file):
        print(f"Lỗi: Không tìm thấy file {input_file}")
        return

    with open(input_file, 'r', encoding='utf-8') as f:
        names = [line.strip() for line in f if line.strip()]

    for name in names:
        print(f"Đang xử lý: {name}...")
        page = wiki.page(name)

        if page.exists():
            # Lấy toàn bộ cấu trúc các mục bao gồm cả Chú thích/Tham khảo
            all_sections = get_all_sections(page.sections)
            
            # Tìm riêng phần Chú thích để đưa ra ngoài nếu bạn muốn cấu trúc riêng
            notes = next((s['text'] for s in all_sections if "Chú thích" in s['title']), "")

            data = {
                "name": name,
                "title": page.title,
                "summary": page.summary.strip(),
                "full_content": page.text.strip(),
                "notes": notes, # Lưu riêng phần Chú thích
                "url": page.fullurl,
                "structure": all_sections # Lưu toàn bộ cấu trúc phân cấp
            }

            results.append(data)
        else:
            print(f"(!) Không tìm thấy trang Wikipedia cho: {name}")

    with open(output_file, 'w', encoding='utf-8') as jf:
        json.dump(results, jf, ensure_ascii=False, indent=4)

    print(f"\nHoàn thành! Đã kiểm tra cả phần Chú thích.")

if __name__ == "__main__":
    process_historical_figures("danh_sach_nhan_vat.txt", "ket_qua_chi_tiet.json")

Đang xử lý: Đinh Tiên Hoàng...
Đang xử lý: Hồ Chí Minh...
Đang xử lý: Hai Bà Trưng...
Đang xử lý: Hùng Vương...
Đang xử lý: Lê Thái Tổ...
Đang xử lý: Lê Đại Hành...
Đang xử lý: Lý Nam Đế...
Đang xử lý: Lý Thái Tổ...
Đang xử lý: Lý Thường Kiệt...
Đang xử lý: Quang Trung...
Đang xử lý: Ngô Quyền...
Đang xử lý: Nguyễn Trãi...
Đang xử lý: Trần Hưng Đạo...
Đang xử lý: Trần Nhân Tông...

Hoàn thành! Đã kiểm tra cả phần Chú thích.


In [ ]:
import wikipediaapi
import requests
from bs4 import BeautifulSoup
import json
import os

def get_citations_by_scraping(url):
    """Sử dụng BeautifulSoup để lấy nội dung trong mục Chú thích/Tham khảo"""
    try:
        response = requests.get(url)
        soup = BeautifulSoup(response.content, 'html.parser')
        citations = []
        
        # Wikipedia thường dùng class 'references' hoặc 'reflist' cho mục Chú thích
        ref_list = soup.find('ol', {'class': 'references'}) or soup.find('div', {'class': 'reflist'})
        
        if ref_list:
            for li in ref_list.find_all('li'):
                # Lấy text và loại bỏ các ký tự điều hướng hệ thống (^, a, b, c...)
                text = li.get_text(separator=" ").strip()
                # Xử lý làm sạch các dấu ^ ở đầu dòng
                clean_text = text.lstrip('^ ').strip()
                citations.append(clean_text)
        return citations
    except:
        return []

def get_all_sections_recursive(sections):
    """Giữ nguyên hàm đệ quy để bảo toàn cấu trúc phân cấp nội dung"""
    sect_data = []
    for s in sections:
        # Loại bỏ các mục hệ thống khỏi cây nội dung nếu muốn, 
        # nhưng ở đây ta giữ lại để đảm bảo tính đầy đủ
        sect_data.append({
            "title": s.title,
            "text": s.text.strip(),
            "subsections": get_all_sections_recursive(s.sections)
        })
    return sect_data

def process_historical_figures(input_file, output_file):
    wiki = wikipediaapi.Wikipedia(
        user_agent='HistoricalNLPProject/1.0',
        language='vi',
        extract_format=wikipediaapi.ExtractFormat.WIKI
    )

    results = []

    if not os.path.exists(input_file):
        print(f"Không tìm thấy file: {input_file}")
        return

    with open(input_file, 'r', encoding='utf-8') as f:
        names = [line.strip() for line in f if line.strip()]

    for name in names:
        print(f"Đang xử lý: {name}...")
        page = wiki.page(name)

        if page.exists():
            # 1. Lấy cấu trúc nội dung đầy đủ (Phân cấp)
            content_structure = get_all_sections_recursive(page.sections)
            
            # 2. Lấy danh sách chú thích chi tiết bằng Scraping
            citations = get_citations_by_scraping(page.fullurl)

            # 3. Tổng hợp vào JSON
            data = {
                "name": name,
                "title": page.title,
                "summary": page.summary.strip(),
                "full_content_text": page.text.strip(), # Nội dung phẳng
                "content_hierarchy": content_structure, # Nội dung phân cấp
                "citations": citations,                 # Danh sách chú thích sạch
                "url": page.fullurl
            }
            results.append(data)
        else:
            print(f"(!) Không tìm thấy trang: {name}")

    with open(output_file, 'w', encoding='utf-8') as jf:
        json.dump(results, jf, ensure_ascii=False, indent=4)

    print(f"\nĐã hoàn thành! Kết quả lưu tại: {output_file}")

if __name__ == "__main__":
    # Đảm bảo bạn có file danh_sach.txt chứa tên nhân vật (ví dụ: Đinh Tiên Hoàng)
    process_historical_figures("danh_sach_nhan_vat.txt", "data_lich_su_chi_tiet.json")

Đang xử lý: Đinh Tiên Hoàng...
Đang xử lý: Hồ Chí Minh...
Đang xử lý: Hai Bà Trưng...
Đang xử lý: Hùng Vương...
Đang xử lý: Lê Thái Tổ...
Đang xử lý: Lê Đại Hành...
Đang xử lý: Lý Nam Đế...
Đang xử lý: Lý Thái Tổ...
Đang xử lý: Lý Thường Kiệt...
Đang xử lý: Quang Trung...
Đang xử lý: Ngô Quyền...
Đang xử lý: Nguyễn Trãi...
Đang xử lý: Trần Hưng Đạo...
Đang xử lý: Trần Nhân Tông...

Đã hoàn thành! Kết quả lưu tại: data_lich_su_chi_tiet.json


: 